In [ ]:
# =============================================================================
# LOCAL ENVIRONMENT BOOTSTRAP (machine-specific paths live in .env, not here)
# =============================================================================
import os
import sys
from pathlib import Path

# Load .env from the repo root so the notebook also works outside VS Code
# (VS Code applies python.envFile itself, so this is just a safety net).
try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None
if load_dotenv is not None:
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        if (_candidate / ".env").is_file():
            load_dotenv(_candidate / ".env")
            break

# Fallback used when AO_CODE_PATHS is not set (the original network-drive setup).
_DEFAULT_AO_CODE_PATHS = [
    r"Z:\Public\Dirk\CodeProjects\OffDesk Assignment\quantresearch",
    r"Z:\Public\Dirk\CodeProjects\OffDesk Assignment\ao_common",
    r"Z:\Public\Dirk\CodeProjects\OffDesk Assignment",
]

_raw_ao_paths = os.environ.get("AO_CODE_PATHS", "")
AO_CODE_PATHS = [
    Path(p.strip()).expanduser()
    for p in (_raw_ao_paths.split(";") if _raw_ao_paths else _DEFAULT_AO_CODE_PATHS)
    if p.strip()
]

# Where mm_stats_data.xlsx / *_trades_data.csv live and where output is written.
DATA_DIR = Path(os.environ.get("MONTHLY_REPORT_DATA_DIR") or Path.cwd()).expanduser().resolve()

for _path in reversed(AO_CODE_PATHS):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

print("ao code paths:")
for _path in AO_CODE_PATHS:
    print(f"  [{'ok ' if _path.exists() else 'MISSING'}] {_path}")
print(f"data dir: {DATA_DIR} [{'ok' if DATA_DIR.exists() else 'MISSING'}]")

_missing = [p for p in AO_CODE_PATHS if not p.exists()]
if _missing:
    print(
        "\nWARNING: the paths above marked MISSING do not exist on this machine, so "
        "`import ao` will fail.\nSet AO_CODE_PATHS in .env to your local clones of "
        "quantresearch / ao_common (see .env.example)."
    )


# =============================================================================
# USER INPUTS (ONLY CHANGE THESE 4 OPTIONS)
# =============================================================================
from datetime import date as _date, datetime as _datetime, timedelta as _timedelta
import matplotlib.patheffects as pe

USER_START_DATE = "2024-01-01"            # Format: YYYY-MM-DD
USER_END_DATE = "2026-08-31"              # Format: YYYY-MM-DD

# USER_SECTORS must be one of the keys in DESK_GROUPS (config cell below). Valid options:
#   "Sectors A"
#   "Sectors B"
#   "Sectors A + B"
#   "COUNTRY"
#   "SSO (excl. Sectors A + B)"
#   "Cars"
#   "Banks"
USER_SECTORS = "SSO"

INCLUDE_CURRENT_MONTH = False              # True = keep current month, False = exclude current month



In [ ]:
def _parse_iso_date(value):
    return _datetime.strptime(value, "%Y-%m-%d").date()


def _last_day_previous_month(value):
    first_day_this_month = value.replace(day=1)
    return first_day_this_month - _timedelta(days=1)


_effective_start_date = _parse_iso_date(USER_START_DATE)
_effective_end_date = _parse_iso_date(USER_END_DATE)

if not INCLUDE_CURRENT_MONTH:
    _today = _date.today()
    if _effective_end_date.year == _today.year and _effective_end_date.month == _today.month:
        _effective_end_date = _last_day_previous_month(_effective_end_date)

if _effective_end_date < _effective_start_date:
    raise ValueError("USER_END_DATE must be on or after USER_START_DATE after applying INCLUDE_CURRENT_MONTH")

SECTOR_TO_PORTFOLIOS = {
    "Sectors A": ["SSO SECTORS A"],
    "Sectors B": ["SSO SECTORS B"],
    "Sectors A + B": ["SSO SECTORS A", "SSO SECTORS B"],
    "COUNTRY": ["COUNTRY"],
    "Cars": ["SSO CARS"],
    "Banks": ["SSO BANKS EU"],
    "SSO (excl. Sectors A + B)": ["SSO (excl. Sectors A + B)"],
}

PORTFOLIO_NAME_FOR_OVERLAY = {
    "Sectors A": "SSO SECTORS A",
    "Sectors B": "SSO SECTORS B",
    "Sectors A + B": "SSO SECTORS A + B",
    "COUNTRY": "COUNTRY",
    "Cars": "SSO CARS",
    "Banks": "SSO BANKS EU",
}

# =============================================================================
# LOAD MM STATS FROM EXCEL
# =============================================================================

import pandas as pd
from pathlib import Path

MM_STATS_FILE = DATA_DIR / "mm_stats_data.xlsx"

PROCESSED_PROF_FILE = DATA_DIR / 'prof_trades_processed.parquet'
PROCESSED_SCREEN_FILE = DATA_DIR / 'screen_trades_processed.parquet'


def _load_mm_stats_sheet(path, sheet_name):
    """Read one sheet from mm_stats_data.xlsx and return {portfolio: {month: value}} dict."""
    df = pd.read_excel(path, sheet_name=sheet_name, index_col=0)
    result = {}
    for col in df.columns:
        result[col] = {idx: val for idx, val in df[col].items() if pd.notna(val)}
    return result


MS_DATA = _load_mm_stats_sheet(MM_STATS_FILE, "MS")
VEGA_SHARE_DATA = _load_mm_stats_sheet(MM_STATS_FILE, "Vega Share")
MARGIN_REV_DATA = _load_mm_stats_sheet(MM_STATS_FILE, "Margin Rev")

print(f"Loaded mm_stats from {MM_STATS_FILE}")
print(f"  MS portfolios: {list(MS_DATA.keys())}")
print(f"  Vega Share portfolios: {list(VEGA_SHARE_DATA.keys())}")
print(f"  Margin Rev portfolios: {list(MARGIN_REV_DATA.keys())}")

# =============================================================================
# CONFIGURATION SECTION
# =============================================================================

# System Paths (resolved in the bootstrap cell from AO_CODE_PATHS / .env)
SYSTEM_PATHS = [str(p) for p in AO_CODE_PATHS]

# Data Provider Settings
ENVIRONMENT = 'prod'
VERBOSE = False
FILL_GAPS = False

# File Paths
PROF_TRADES_FILE = DATA_DIR / 'prof_trades_data.csv'
SCREEN_TRADES_FILE = DATA_DIR / 'screen_trades_data.csv'

# Date Ranges (derived from user inputs)
PROF_START_DATE = (_effective_start_date.year, _effective_start_date.month, _effective_start_date.day)
PROF_END_DATE = (_effective_end_date.year, _effective_end_date.month, _effective_end_date.day)
SCREEN_START_DATE = (_effective_start_date.year, _effective_start_date.month, _effective_start_date.day)
SCREEN_END_DATE = (_effective_end_date.year, _effective_end_date.month, _effective_end_date.day)
FILTER_START_DATE = _effective_start_date.isoformat()
FILTER_END_DATE = (_effective_end_date + _timedelta(days=1)).isoformat()

# Plot x-axis limits with padding
_xlim_start_other = pd.Timestamp(FILTER_START_DATE) - pd.Timedelta(days=15)
_xlim_end_other = pd.Timestamp(_effective_end_date) + pd.Timedelta(days=15)

_xlim_start_weekly = pd.Timestamp(FILTER_START_DATE) - pd.Timedelta(days=10)
_xlim_end_weekly = pd.Timestamp(_effective_end_date) + pd.Timedelta(days=8)
_xlim_start_monthly = pd.Timestamp(FILTER_START_DATE) - pd.Timedelta(days=20)
_xlim_end_monthly = pd.Timestamp(_effective_end_date) + pd.Timedelta(days=-3)

_plot_months = pd.date_range(start=FILTER_START_DATE, end=str(_effective_end_date), freq='MS')

# Portfolio Configuration (derived from user sector input)
SELECTED_DESK_GROUP = USER_SECTORS
PORTFOLIO_NAMES = SECTOR_TO_PORTFOLIOS.get(USER_SECTORS, [USER_SECTORS])
FIGURE_TITLE_PORTFOLIO_NAME = " + ".join(PORTFOLIO_NAMES)

# Desk-based filtering configuration
USE_DESK_FILTER = True
DESK_GROUPS = {
    "Sectors A": [
        "Construction", "Consumer", "Consumer Staples",
        "Sector Other", "Telecom"
    ],
    "Sectors B": [
        "Banks", "Cars", "Insurance", "Semiconductors"
    ],
    "Sectors A + B": [
        "Construction", "Consumer", "Consumer Staples",
        "Sector Other", "Telecom",
        "Banks", "Cars", "Insurance", "Semiconductors"
    ],
    "COUNTRY": "ALL",
    "SSO (excl. Sectors A + B)": {"exclude": [
        "Construction", "Consumer", "Consumer Staples",
        "Sector Other", "Telecom",
        "Banks", "Cars", "Insurance", "Semiconductors"
    ]},
    "Cars": ["Cars"],
    "Banks": ["Banks"]
}

HIERARCHY_DATE = (2026, 1, 28)
HIERARCHY_DATE_SCREEN = (2025, 11, 28)
EXCLUDE_DESK = "Index Passive"

# Excel Export Configuration
SAVE_EXCEL = True
EXCEL_OUTPUT_DIR = str(DATA_DIR / 'output_data')

# Screen Trades Origins
SCREEN_TRADES_ORIGINS = [
    'Manual', 'FastHiddens', 'Morning', 'Direct Trade',
    'AQ', 'Volatility Order', 'Vega Order'
]

# Analysis Settings
COMBO_TIME_THRESHOLD = 10
MAX_COMBO_LEGS_PROF = 3
MAX_COMBO_LEGS_SCREEN = 2
SELECTED_METRIC = 'nvedi'
ROLLING_WINDOW_DAYS = 30

# Plotting Settings
WEEKLY_BAR_WIDTH = 5
MONTHLY_BAR_WIDTH = 20

# Plot Saving Configuration
SAVE_PLOTS = True
PLOT_OUTPUT_DIR = str(DATA_DIR / 'output_plots')
PLOT_FORMAT = 'png'
PLOT_DPI = 300

import sys
for path in SYSTEM_PATHS:
    if path not in sys.path:
        sys.path.insert(0, path)

import ao
import importlib.util
import sys

# Handle pkg_resources import issue
try:
    import pkg_resources
except ModuleNotFoundError:
    # pkg_resources is part of setuptools; try to use importlib as fallback
    spec = importlib.util.find_spec("setuptools")
    if spec is not None:
        import setuptools
        import pkg_resources
    else:
        print("Warning: pkg_resources not available, some features may not work")
print(ao.__file__)

# =============================================================================
# IMPORTS AND SETUP
# =============================================================================

import uuid
import sys
import os
print(sys.path)

from datetime import datetime
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import date as Date, timedelta as TimeDelta
from ao.quantlib import DataProvider
from ao.quantlib.utils import get_leaf_portfolios_from_struct
import warnings
warnings.filterwarnings('ignore')
import setuptools
import pkg_resources

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def save_plot(fig, filename):
    if SAVE_PLOTS:
        if not os.path.exists(PLOT_OUTPUT_DIR):
            os.makedirs(PLOT_OUTPUT_DIR)
        filepath = os.path.join(PLOT_OUTPUT_DIR, f"{filename}.{PLOT_FORMAT}")
        fig.savefig(filepath, dpi=PLOT_DPI, bbox_inches='tight')
        print(f"Plot saved: {filepath}")

def extract_uuid_timestamp(uuid_str):
    try:
        u = uuid.UUID(str(uuid_str))
        if u.version == 1:
            timestamp = (u.time - 0x01b21dd213814000) / 1e7
            return datetime.fromtimestamp(timestamp)
    except:
        return None
    return None

def save_excel(df, filename, sheet_name='Data'):
    if SAVE_EXCEL:
        if not os.path.exists(EXCEL_OUTPUT_DIR):
            os.makedirs(EXCEL_OUTPUT_DIR)
        filepath = os.path.join(EXCEL_OUTPUT_DIR, f"{filename}.xlsx")
        df.to_excel(filepath, sheet_name=sheet_name, index=False)
        print(f"Excel saved: {filepath}")

def identify_combo_type(group):
    group_with_strike = group[group['strike'].notna()]
    group_with_strike = group_with_strike[group_with_strike['strike'] != 0]

    if len(group_with_strike) == 1:
        return None

    if len(group_with_strike) == 2:
        leg1, leg2 = group_with_strike.iloc[0], group_with_strike.iloc[1]

        same_type = leg1['is_call'] == leg2['is_call']
        same_side = leg1['side_int'] == leg2['side_int']
        same_maturity = leg1['maturity_date'] == leg2['maturity_date']
        same_strike = leg1['strike'] == leg2['strike']

        if same_strike and same_maturity and not same_type and same_side:
            return "Straddle"
        elif not same_strike and same_maturity and not same_type and same_side:
            return "Strangle"
        elif same_type and same_maturity and not same_side:
            option_type = "Call" if leg1['is_call'] else "Put"
            return f"{option_type} Spread"
        elif same_type and not same_maturity and not same_side:
            option_type = "Call" if leg1['is_call'] else "Put"
            return f"{option_type} Calendar"
        elif not same_type and not same_side and same_maturity:
            return "Risk Reversal"
        else:
            return "Other"

    if len(group_with_strike) >= 3:
        return "Other"

    return None

def calculate_combo_metric(group, metric_col):
    group_with_strike = group[group['strike'].notna()]
    group_with_strike = group_with_strike[group_with_strike['strike'] != 0]

    if len(group_with_strike) == 0:
        return None, None

    combo_name = group['combo_name'].iloc[0]

    if pd.isna(combo_name):
        if len(group_with_strike) == 1:
            return group_with_strike[metric_col].iloc[0], group_with_strike.index[0]
        return group_with_strike[metric_col].sum(), group_with_strike.index[0]

    if combo_name in ['Call Calendar', 'Put Calendar']:
        metric1 = group_with_strike[metric_col].iloc[0]
        metric2 = group_with_strike[metric_col].iloc[1]

        if abs(metric1) > abs(metric2):
            return metric1, group_with_strike.index[0]
        else:
            return metric2, group_with_strike.index[1]

    return group_with_strike[metric_col].sum(), group_with_strike.index[0]

def aggregate_by_period(df, metric_col, period='W'):
    df_copy = df.copy()
    df_copy['period'] = df_copy['trade_date'].dt.to_period(period)

    result = df_copy.groupby('period').agg({
        metric_col: lambda x: x.abs().sum()
    }).reset_index()

    result['period_start'] = result['period'].dt.start_time
    return result

# =============================================================================
# 1. LOAD PROFESSIONAL TRADES DATA
# =============================================================================

print("="*80)
print("LOADING PROFESSIONAL TRADES DATA")
print("="*80)

dp = DataProvider(env=ENVIRONMENT, verbose=VERBOSE)

start_date = Date(*PROF_START_DATE)
end_date = Date(*PROF_END_DATE)

def get_eu_holidays(year):
    holidays = [
        Date(year, 1, 1),
        Date(year, 12, 25),
        Date(year, 12, 26),
    ]

    good_friday_dates = {
        2024: Date(2024, 3, 29),
        2025: Date(2025, 4, 18),
        2026: Date(2026, 4, 3),
        2027: Date(2027, 3, 26),
        2028: Date(2028, 4, 14),
    }
    if year in good_friday_dates:
        holidays.append(good_friday_dates[year])

    easter_monday_dates = {
        2024: Date(2024, 4, 1),
        2025: Date(2025, 4, 21),
        2026: Date(2026, 4, 6),
        2027: Date(2027, 3, 29),
        2028: Date(2028, 4, 17),
    }
    if year in easter_monday_dates:
        holidays.append(easter_monday_dates[year])

    return holidays

eu_holidays = set()
for year in range(start_date.year, end_date.year + 1):
    eu_holidays.update(get_eu_holidays(year))

print(f"Loaded {len(eu_holidays)} EU-wide holidays to skip")

if os.path.exists(PROF_TRADES_FILE):
    prof_trades_df = pd.read_csv(PROF_TRADES_FILE, parse_dates=['trade_time'])
    prof_trades_df['trade_time'] = pd.to_datetime(prof_trades_df['trade_time'])
    existing_dates = prof_trades_df['trade_time'].dt.date
    first_existing = existing_dates.min()
    last_existing = existing_dates.max()
    print(f"Existing data spans {first_existing} to {last_existing} ({len(prof_trades_df)} trades)")
else:
    prof_trades_df = pd.DataFrame()
    first_existing = None
    last_existing = None
    print("No existing data found")

dates_to_fetch = []
current_date = start_date
while current_date <= end_date:
    if current_date.weekday() < 5 and current_date not in eu_holidays:
        if first_existing is None:
            dates_to_fetch.append(current_date)
        elif current_date < first_existing or current_date > last_existing:
            dates_to_fetch.append(current_date)
    current_date = current_date + TimeDelta(days=1)

print(f"Need to fetch {len(dates_to_fetch)} new dates")

if dates_to_fetch:
    prof_trades_list = []

    with tqdm(total=len(dates_to_fetch), desc="Loading new prof trades") as pbar:
        for fetch_date in dates_to_fetch:
            print(f"Fetchin for date: {fetch_date}")
            try:
                our_trades_df = dp.get_trades(date=fetch_date)

                if our_trades_df.empty:
                    pbar.update(1)
                    continue

                if 'comment' not in our_trades_df.columns:
                    print(f"\nWarning: 'comment' column not found for {fetch_date}, skipping")
                    pbar.update(1)
                    continue

                prof_trades = our_trades_df[our_trades_df['comment'] == 'Prof Trade']

                if not prof_trades.empty:
                    prof_trades = prof_trades.reset_index(names='trade_id')
                    prof_trades_list.append(prof_trades)

            except KeyError as e:
                print(f"\nKeyError fetching {fetch_date}: {e}")
            except Exception as e:
                print(f"\nError fetching {fetch_date}: {e}")
            pbar.update(1)

    if prof_trades_list:
        new_data = pd.concat(prof_trades_list, ignore_index=True)

        portfolio_names_df = dp.get_portfolios(portfolio_ids=new_data["portfolio_id"].unique().tolist())
        new_data['portfolio_name'] = new_data['portfolio_id'].map(portfolio_names_df['name'])
        new_data['trade_time'] = new_data['trade_id'].apply(extract_uuid_timestamp)

        prof_trades_df = pd.concat([prof_trades_df, new_data], ignore_index=True)
        prof_trades_df.to_csv(PROF_TRADES_FILE, index=False)
        print(f"Added {len(new_data)} new prof trades")
    else:
        print("No new prof trades found for the fetched dates")
else:
    print("Prof trades data is complete")

# =============================================================================
# 2. PROCESS PROFESSIONAL TRADES
# =============================================================================

print("\n" + "="*80)
print("PROCESSING PROFESSIONAL TRADES")
print("="*80)

def _needs_reprocessing(processed_path, raw_path):
    if not os.path.exists(processed_path):
        return True
    if not os.path.exists(raw_path):
        return True
    return os.path.getmtime(raw_path) > os.path.getmtime(processed_path)

if _needs_reprocessing(PROCESSED_PROF_FILE, PROF_TRADES_FILE):
    print("Processing prof trades from scratch (raw data newer or no cache)...")

    prof_trades_df = pd.read_csv(PROF_TRADES_FILE, parse_dates=['trade_time'])
    prof_trades_df['trade_time'] = pd.to_datetime(prof_trades_df['trade_time'])
    prof_trades_df = prof_trades_df[prof_trades_df['desk'] != EXCLUDE_DESK]

    fx_rates = dp.get_fx_rates(date=Date.today(), return_type="dict")
    prof_trades_df['to_euro'] = prof_trades_df['currency_code'].map(fx_rates)

    if "maturity_date" in prof_trades_df.columns:
        prof_trades_df['dtm'] = prof_trades_df.apply(
            lambda row: np.busday_count(row['trade_time'].date(), row['maturity_date'])
            if pd.notnull(row['maturity_date']) else np.nan,
            axis=1
        )
        prof_trades_df["trade_vega_euro"] = prof_trades_df['trade_vega'] * prof_trades_df['to_euro']
        prof_trades_df["trade_vedi_euro"] = (
            np.minimum(np.sqrt(20 / prof_trades_df["dtm"]), 3) * prof_trades_df["trade_vega_euro"]
        )
        prof_trades_df["trade_nvedi_euro"] = (
            prof_trades_df["trade_vedi_euro"] * prof_trades_df['trade_volatility']
        )
    else:
        prof_trades_df["dtm"] = np.nan
        prof_trades_df["trade_vedi_euro"] = np.nan
        prof_trades_df["trade_nvedi_euro"] = np.nan

    prof_trades_df = prof_trades_df.sort_values('trade_time').reset_index(drop=True)

    time_diff = prof_trades_df['trade_time'].diff().dt.total_seconds()
    portfolio_changed = prof_trades_df['portfolio_name'] != prof_trades_df['portfolio_name'].shift(1)
    prof_trades_df['combo_group'] = (
        (time_diff > COMBO_TIME_THRESHOLD) | portfolio_changed
    ).fillna(True).cumsum()

    prof_trades_df['is_call'] = prof_trades_df['name'].str.contains(' C ')

    has_strike = prof_trades_df['strike'].notna()
    combo_leg_counts = prof_trades_df[has_strike].groupby('combo_group').size()
    multi_leg_combos = combo_leg_counts[combo_leg_counts > MAX_COMBO_LEGS_PROF]

    if len(multi_leg_combos) > 0:
        print(f"Found {len(multi_leg_combos)} combo groups with more than {MAX_COMBO_LEGS_PROF} legs")
    else:
        print(f"All combos have {MAX_COMBO_LEGS_PROF} or fewer legs")

    combo_names = prof_trades_df.groupby('combo_group').apply(identify_combo_type)
    prof_trades_df['combo_name'] = prof_trades_df['combo_group'].map(combo_names)

    for metric in ['vega', 'vedi', 'nvedi']:
        trade_col = f'trade_{metric}_euro'
        combo_col = f'combo_{metric}_euro'

        prof_trades_df[combo_col] = np.nan

        for combo_group in prof_trades_df['combo_group'].unique():
            group = prof_trades_df[prof_trades_df['combo_group'] == combo_group]
            combo_value, target_idx = calculate_combo_metric(group, trade_col)

            if combo_value is not None and target_idx is not None:
                prof_trades_df.loc[target_idx, combo_col] = combo_value

    prof_trades_df.to_parquet(PROCESSED_PROF_FILE, index=False)
    print(f"Saved processed prof trades to {PROCESSED_PROF_FILE}")
else:
    print(f"Loading cached processed prof trades from {PROCESSED_PROF_FILE}...")
    prof_trades_df = pd.read_parquet(PROCESSED_PROF_FILE)

prof_trades_df = prof_trades_df[prof_trades_df['trade_time'] >= FILTER_START_DATE]
prof_trades_df = prof_trades_df[prof_trades_df['trade_time'] < FILTER_END_DATE]

print(f"Prof trades after date filter: {len(prof_trades_df)}")

# =============================================================================
# 3. FILTER PROFESSIONAL TRADES BY PORTFOLIO OR DESK
# =============================================================================

print("\n" + "="*80)
print("FILTERING BY PORTFOLIO OR DESK")
print("="*80)

if USE_DESK_FILTER:
    desk_list = DESK_GROUPS.get(SELECTED_DESK_GROUP)

    if desk_list == "ALL":
        prof_trades_group = prof_trades_df.copy()
        filter_name = "All Desks (COUNTRY)"
    elif isinstance(desk_list, dict) and "exclude" in desk_list:
        prof_trades_group = prof_trades_df[~prof_trades_df['desk'].isin(desk_list["exclude"])]
        filter_name = SELECTED_DESK_GROUP
    else:
        prof_trades_group = prof_trades_df[prof_trades_df['desk'].isin(desk_list)]
        filter_name = SELECTED_DESK_GROUP

    print(f"Filtered to {len(prof_trades_group)} prof trades for {filter_name}")
    FIGURE_TITLE_PORTFOLIO_NAME = filter_name

else:
    struct = dp.redis_dp.get_current_portfolio_hierarchy()
    pf_id = dp.find_portfolios(names=PORTFOLIO_NAMES)
    leaf_pf_id = get_leaf_portfolios_from_struct(struct, pf_id)
    leaf_pf_id = [str(s) for s in leaf_pf_id]

    prof_trades_group = prof_trades_df[prof_trades_df['portfolio_id'].isin(leaf_pf_id)]
    filter_name = FIGURE_TITLE_PORTFOLIO_NAME
    print(f"Filtered to {len(prof_trades_group)} prof trades for {filter_name}")

save_excel(prof_trades_group, f'prof_trades_{FIGURE_TITLE_PORTFOLIO_NAME}', sheet_name='Professional Trades')

# =============================================================================
# 4. LOAD SCREEN TRADES DATA
# =============================================================================

print("\n" + "="*80)
print("LOADING SCREEN TRADES DATA")
print("="*80)

start_date = Date(*SCREEN_START_DATE)
end_date = Date(*SCREEN_END_DATE)

if os.path.exists(SCREEN_TRADES_FILE):
    screen_trades_df = pd.read_csv(SCREEN_TRADES_FILE, parse_dates=['trade_time'])
    screen_trades_df['trade_time'] = pd.to_datetime(screen_trades_df['trade_time'])
    existing_dates = screen_trades_df['trade_time'].dt.date
    first_existing = existing_dates.min()
    last_existing = existing_dates.max()
    print(f"Existing screen data spans {first_existing} to {last_existing} ({len(screen_trades_df)} trades)")
else:
    screen_trades_df = pd.DataFrame()
    first_existing = None
    last_existing = None

dates_to_fetch = []
current_date = start_date
while current_date <= end_date:
    if FILL_GAPS:
        existing_day_dates = set(screen_trades_df['trade_time'].dt.date) if not screen_trades_df.empty else set()
        if current_date.weekday() < 5 and current_date not in eu_holidays:
            if current_date not in existing_day_dates:
                dates_to_fetch.append(current_date)
    else:
        if current_date.weekday() < 5 and current_date not in eu_holidays:
            if first_existing is None:
                dates_to_fetch.append(current_date)
            elif current_date < first_existing or current_date > last_existing:
                dates_to_fetch.append(current_date)
    current_date = current_date + TimeDelta(days=1)

if dates_to_fetch:
    screen_trades_list = []

    with tqdm(total=len(dates_to_fetch), desc="Loading new screen trades") as pbar:
        for fetch_date in dates_to_fetch:
            try:
                our_trades_df = dp.get_trades(date=fetch_date)
                screen_trades = our_trades_df[our_trades_df['origin'].isin(SCREEN_TRADES_ORIGINS)]
                screen_trades = screen_trades[screen_trades['comment'] != 'Prof Trade']
                screen_trades = screen_trades[screen_trades['desk'] != EXCLUDE_DESK]

                if not screen_trades.empty:
                    screen_trades = screen_trades.reset_index(names='trade_id')
                    screen_trades_list.append(screen_trades)
            except:
                pass
            pbar.update(1)

    if screen_trades_list:
        new_data = pd.concat(screen_trades_list, ignore_index=True)

        portfolio_names_df = dp.get_portfolios(portfolio_ids=new_data["portfolio_id"].unique().tolist())
        new_data['portfolio_name'] = new_data['portfolio_id'].map(portfolio_names_df['name'])
        new_data['trade_time'] = new_data['trade_id'].apply(extract_uuid_timestamp)

        screen_trades_df = pd.concat([screen_trades_df, new_data], ignore_index=True)
        screen_trades_df.to_csv(SCREEN_TRADES_FILE, index=False)
        print(f"Added {len(new_data)} new screen trades")
    else:
        print("No new screen trades found")
else:
    print("Screen trades data is complete")

# =============================================================================
# 5. PROCESS SCREEN TRADES
# =============================================================================

print("\n" + "="*80)
print("PROCESSING SCREEN TRADES")
print("="*80)

if _needs_reprocessing(PROCESSED_SCREEN_FILE, SCREEN_TRADES_FILE):
    print("Processing screen trades from scratch (raw data newer or no cache)...")

    screen_trades_df = pd.read_csv(SCREEN_TRADES_FILE)
    screen_trades_df['trade_time'] = pd.to_datetime(screen_trades_df['trade_time'])
    screen_trades_df = screen_trades_df[screen_trades_df['desk'] != EXCLUDE_DESK]
    screen_trades_df = screen_trades_df[screen_trades_df['product_type'] != 'future']

    fx_rates = dp.get_fx_rates(date=Date.today(), return_type="dict")
    screen_trades_df['to_euro'] = screen_trades_df['currency_code'].map(fx_rates)

    if "maturity_date" in screen_trades_df.columns:
        screen_trades_df['maturity_date'] = pd.to_datetime(screen_trades_df['maturity_date'])

        mask = screen_trades_df['maturity_date'].notna()
        screen_trades_df['dtm'] = np.nan

        screen_trades_df.loc[mask, 'dtm'] = np.busday_count(
            screen_trades_df.loc[mask, 'trade_time'].dt.date.values.astype('datetime64[D]'),
            screen_trades_df.loc[mask, 'maturity_date'].dt.date.values.astype('datetime64[D]')
        )

        screen_trades_df["trade_vega_euro"] = screen_trades_df['trade_vega'] * screen_trades_df['to_euro']
        screen_trades_df["trade_vedi_euro"] = (
            np.minimum(np.sqrt(20 / screen_trades_df["dtm"]), 3) * screen_trades_df["trade_vega_euro"]
        )
        screen_trades_df["trade_nvedi_euro"] = (
            screen_trades_df["trade_vedi_euro"] * screen_trades_df['trade_volatility']
        )
    else:
        screen_trades_df["dtm"] = np.nan
        screen_trades_df["trade_vedi_euro"] = np.nan
        screen_trades_df["trade_nvedi_euro"] = np.nan

    screen_trades_df = screen_trades_df.sort_values('trade_time').reset_index(drop=True)

    screen_trades_df['is_call'] = screen_trades_df['name'].str.contains(' C ')

    has_strike = screen_trades_df['strike'].notna() & (screen_trades_df['strike'] != 0)
    combo_leg_counts = screen_trades_df[has_strike].groupby('combination_instrument_id').size()
    multi_leg_combos = combo_leg_counts[combo_leg_counts > MAX_COMBO_LEGS_SCREEN]

    if len(multi_leg_combos) > 0:
        print(f"Found {len(multi_leg_combos)} combo groups with more than {MAX_COMBO_LEGS_SCREEN} legs")
    else:
        print(f"All combos have {MAX_COMBO_LEGS_SCREEN} or fewer legs")

    combo_names = screen_trades_df.groupby('combination_instrument_id').apply(identify_combo_type)
    screen_trades_df['combo_name'] = screen_trades_df['combination_instrument_id'].map(combo_names)

    valid_strike_mask = screen_trades_df['strike'].notna() & (screen_trades_df['strike'] != 0)

    for metric in ['vega', 'vedi', 'nvedi']:
        trade_col = f'trade_{metric}_euro'
        combo_col = f'combo_{metric}_euro'

        screen_trades_df[combo_col] = screen_trades_df[trade_col]

        combo_mask = screen_trades_df['combo_name'].notna() & valid_strike_mask
        combo_df = screen_trades_df.loc[combo_mask, ['combination_instrument_id', 'combo_name', trade_col]].copy()

        if len(combo_df) == 0:
            continue

        calendar_mask = combo_df['combo_name'].isin(['Call Calendar', 'Put Calendar'])

        if (~calendar_mask).any():
            non_calendar = combo_df[~calendar_mask].groupby('combination_instrument_id').agg({
                trade_col: 'sum'
            })
            id_to_value = non_calendar[trade_col].to_dict()
            first_rows = combo_df[~calendar_mask].groupby('combination_instrument_id').head(1)
            screen_trades_df.loc[first_rows.index, combo_col] = (
                first_rows['combination_instrument_id'].map(id_to_value).values
            )

        if calendar_mask.any():
            calendar_df = combo_df[calendar_mask].copy()
            calendar_df['abs_metric'] = calendar_df[trade_col].abs()

            idx_max = calendar_df.groupby('combination_instrument_id')['abs_metric'].idxmax()
            idx_max = idx_max.dropna()

            if len(idx_max) > 0:
                screen_trades_df.loc[idx_max, combo_col] = calendar_df.loc[idx_max, trade_col].values

    screen_trades_df.to_parquet(PROCESSED_SCREEN_FILE, index=False)
    print(f"Saved processed screen trades to {PROCESSED_SCREEN_FILE}")
else:
    print(f"Loading cached processed screen trades from {PROCESSED_SCREEN_FILE}...")
    screen_trades_df = pd.read_parquet(PROCESSED_SCREEN_FILE)

screen_trades_df = screen_trades_df[screen_trades_df['trade_time'] >= FILTER_START_DATE]
screen_trades_df = screen_trades_df[screen_trades_df['trade_time'] < FILTER_END_DATE]

if USE_DESK_FILTER:
    desk_list = DESK_GROUPS.get(SELECTED_DESK_GROUP)

    if desk_list == "ALL":
        screen_trades_group = screen_trades_df.copy()
    elif isinstance(desk_list, dict) and "exclude" in desk_list:
        screen_trades_group = screen_trades_df[~screen_trades_df['desk'].isin(desk_list["exclude"])]
    else:
        screen_trades_group = screen_trades_df[screen_trades_df['desk'].isin(desk_list)]

    print(f"Filtered to {len(screen_trades_group)} screen trades for {SELECTED_DESK_GROUP}")
else:
    screen_trades_group = screen_trades_df[screen_trades_df['portfolio_id'].isin(leaf_pf_id)]
    print(f"Filtered to {len(screen_trades_group)} screen trades for {FIGURE_TITLE_PORTFOLIO_NAME}")

# =============================================================================
# 6. PLOT: CUMULATIVE PROF TRADE ACTIVITY
# =============================================================================

print("\n" + "="*80)
print("GENERATING PLOTS")
print("="*80)

import matplotlib.patheffects as pe

combo_agg = prof_trades_group.groupby('combo_group').agg({
    'trade_time': 'first',
    'combo_vega_euro': 'first',
    'combo_vedi_euro': 'first',
    'combo_nvedi_euro': 'first',
    'combo_name': 'first'
}).reset_index()

combo_agg = combo_agg.dropna(subset=[f'combo_{SELECTED_METRIC}_euro'])
combo_agg = combo_agg.sort_values('trade_time').reset_index(drop=True)

combo_agg['cumulative_count'] = range(1, len(combo_agg) + 1)
combo_agg['cumulative_vega'] = combo_agg['combo_vega_euro'].abs().cumsum()
combo_agg['cumulative_vedi'] = combo_agg['combo_vedi_euro'].abs().cumsum()
combo_agg['cumulative_nvedi'] = combo_agg['combo_nvedi_euro'].abs().cumsum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.scatter(combo_agg['trade_time'], combo_agg['cumulative_count'],
            color='tab:blue', s=30, alpha=0.6)
ax1.set_xlabel('Trade Time', fontsize=12)
ax1.set_ylabel('Cumulative Number of Trades', fontsize=12)
ax1.set_title('Cumulative Trade Count', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax1.set_xticks(_plot_months)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

ax2.scatter(combo_agg['trade_time'], combo_agg[f'cumulative_{SELECTED_METRIC}'],
            color='tab:red', s=30, alpha=0.6)
ax2.set_xlabel('Trade Time', fontsize=12)
ax2.set_ylabel(f'Cumulative {SELECTED_METRIC.capitalize()} (000s)', fontsize=12)
ax2.set_title(f'Cumulative {SELECTED_METRIC.capitalize()}', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000):,}'))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax2.set_xticks(_plot_months)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

fig.suptitle(f'Prof Trade Activity - {FIGURE_TITLE_PORTFOLIO_NAME}', fontsize=14, y=1.02)
plt.tight_layout()
for ax in [ax1, ax2]:
    ax.set_xlim(_xlim_start_other, _xlim_end_other)
save_plot(fig, f'01_cumulative_prof_trades_{FIGURE_TITLE_PORTFOLIO_NAME}')
plt.show()

# =============================================================================
# 7. PLOT: MONTHLY PROF TRADE ACTIVITY
# =============================================================================

combo_agg_monthly = prof_trades_group.groupby('combo_group').agg({
    'trade_time': 'first',
    'combo_vega_euro': 'first',
    'combo_vedi_euro': 'first',
    'combo_nvedi_euro': 'first',
    'combo_name': 'first'
}).reset_index()

combo_agg_monthly = combo_agg_monthly.dropna(subset=['combo_vega_euro', 'combo_vedi_euro', 'combo_nvedi_euro'])
combo_agg_monthly = combo_agg_monthly.sort_values('trade_time').reset_index(drop=True)

combo_agg_monthly['year_month'] = combo_agg_monthly['trade_time'].dt.to_period('M')

monthly_agg = combo_agg_monthly.groupby('year_month').agg({
    'combo_group': 'count',
    'combo_vega_euro': lambda x: x.abs().sum(),
    'combo_vedi_euro': lambda x: x.abs().sum(),
    'combo_nvedi_euro': lambda x: x.abs().sum()
}).reset_index()

monthly_agg.columns = ['year_month', 'num_trades', 'total_vega', 'total_vedi', 'total_nvedi']
monthly_agg['month_start'] = monthly_agg['year_month'].dt.to_timestamp()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.bar(monthly_agg['month_start'], monthly_agg['num_trades'],
        width=MONTHLY_BAR_WIDTH, color='tab:blue', alpha=0.7)
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Number of Trades', fontsize=12)
ax1.set_title('Number of Trades per Month', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax1.set_xticks(_plot_months)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

ax2.bar(monthly_agg['month_start'], monthly_agg[f'total_{SELECTED_METRIC}']/1000,
        width=MONTHLY_BAR_WIDTH, color='tab:red', alpha=0.7)
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel(f'Total {SELECTED_METRIC.capitalize()} (000s)', fontsize=12)
ax2.set_title(f'Total {SELECTED_METRIC.capitalize()} per Month', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax2.set_xticks(_plot_months)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

fig.suptitle(f'Monthly Prof Trade Activity - {FIGURE_TITLE_PORTFOLIO_NAME}', fontsize=14, y=1.02)
plt.tight_layout()
for ax in [ax1, ax2]:
    ax.set_xlim(_xlim_start_monthly, _xlim_end_monthly)
save_plot(fig, f'02_monthly_prof_trades_{FIGURE_TITLE_PORTFOLIO_NAME}')
plt.show()

# =============================================================================
# 8. PREPARE DATA FOR COMPARISON
# =============================================================================

metric_col = f'combo_{SELECTED_METRIC}_euro'
metric_display = SELECTED_METRIC.capitalize()

prof_combo_agg = prof_trades_group.groupby('combo_group').agg({
    'trade_time': 'first',
    metric_col: 'first',
    'combo_name': 'first'
}).reset_index()

prof_combo_agg = prof_combo_agg.dropna(subset=[metric_col])
prof_combo_agg['trade_date'] = pd.to_datetime(prof_combo_agg['trade_time'].dt.date)

screen_trades_group['temp_id'] = screen_trades_group['combination_instrument_id'].fillna(
    screen_trades_group.index.to_series().apply(lambda x: f'single_{x}')
)

screen_combo_agg = screen_trades_group.groupby('temp_id').agg({
    'trade_time': 'first',
    metric_col: 'first',
    'combo_name': 'first'
}).reset_index()

screen_combo_agg = screen_combo_agg.dropna(subset=[metric_col])
screen_combo_agg['trade_date'] = pd.to_datetime(screen_combo_agg['trade_time'].dt.date)

# =============================================================================
# 9. PLOT: ROLLING WINDOW PARTICIPATION
# =============================================================================

prof_daily = prof_combo_agg.groupby('trade_date').agg({
    metric_col: lambda x: x.abs().sum()
}).reset_index()

screen_daily = screen_combo_agg.groupby('trade_date').agg({
    metric_col: lambda x: x.abs().sum()
}).reset_index()

all_dates = sorted(set(prof_daily['trade_date'].tolist() + screen_daily['trade_date'].tolist()))

combined_data = []
for date in all_dates:
    prof_row = prof_daily[prof_daily['trade_date'] == date]
    screen_row = screen_daily[screen_daily['trade_date'] == date]

    prof_value = prof_row[metric_col].iloc[0] if len(prof_row) > 0 else 0
    screen_value = screen_row[metric_col].iloc[0] if len(screen_row) > 0 else 0

    combined_data.append({
        'trade_date': date,
        'prof_value': prof_value,
        'screen_value': screen_value
    })

combined_df = pd.DataFrame(combined_data)
combined_df['trade_date'] = pd.to_datetime(combined_df['trade_date'])

combined_df['prof_rolling'] = combined_df['prof_value'].rolling(window=ROLLING_WINDOW_DAYS, min_periods=1).sum()
combined_df['screen_rolling'] = combined_df['screen_value'].rolling(window=ROLLING_WINDOW_DAYS, min_periods=1).sum()
combined_df['total_rolling'] = combined_df['prof_rolling'] + combined_df['screen_rolling']
combined_df['prof_pct'] = (combined_df['prof_rolling'] / combined_df['total_rolling'] * 100).fillna(0)
combined_df['screen_pct'] = (combined_df['screen_rolling'] / combined_df['total_rolling'] * 100).fillna(0)

fig, ax = plt.subplots(1, 1, figsize=(14, 6))

ax.fill_between(combined_df['trade_date'], 0, combined_df['prof_pct'],
                alpha=0.7, color='tab:blue', label='Professional')
ax.fill_between(combined_df['trade_date'], combined_df['prof_pct'], 100,
                alpha=0.7, color='tab:red', label='Screen')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title(f'{ROLLING_WINDOW_DAYS}-Day Rolling {metric_display} (%)', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 100)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.set_xticks(_plot_months)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle(f'Prof Trading Participation ({metric_display}) - {FIGURE_TITLE_PORTFOLIO_NAME}',
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
ax.set_xlim(_xlim_start_weekly, _xlim_end_weekly)
save_plot(fig, f'03_rolling_participation_{FIGURE_TITLE_PORTFOLIO_NAME}')
plt.show()

# =============================================================================
# 10. PLOT: WEEKLY AND MONTHLY WITH DUAL AXIS
# =============================================================================

prof_weekly = aggregate_by_period(prof_combo_agg, metric_col, period='W')
screen_weekly = aggregate_by_period(screen_combo_agg, metric_col, period='W')

weekly_merged = pd.merge(
    prof_weekly[['period_start', metric_col]],
    screen_weekly[['period_start', metric_col]],
    on='period_start',
    how='outer',
    suffixes=('_prof', '_screen')
).fillna(0)

weekly_merged['total'] = weekly_merged[f'{metric_col}_prof'] + weekly_merged[f'{metric_col}_screen']
weekly_merged['prof_pct'] = (weekly_merged[f'{metric_col}_prof'] / weekly_merged['total'] * 100).fillna(0)

current_week = pd.Timestamp.now().to_period('W').start_time
weekly_merged['is_current'] = weekly_merged['period_start'] == current_week

prof_monthly = aggregate_by_period(prof_combo_agg, metric_col, period='M')
screen_monthly = aggregate_by_period(screen_combo_agg, metric_col, period='M')

monthly_merged = pd.merge(
    prof_monthly[['period_start', metric_col]],
    screen_monthly[['period_start', metric_col]],
    on='period_start',
    how='outer',
    suffixes=('_prof', '_screen')
).fillna(0)

all_months = pd.date_range(start=FILTER_START_DATE,
                           end=str(_effective_end_date), freq='MS')
all_months_df = pd.DataFrame({'period_start': all_months})
monthly_merged = pd.merge(all_months_df, monthly_merged, on='period_start', how='left').fillna(0)

monthly_merged['total'] = monthly_merged[f'{metric_col}_prof'] + monthly_merged[f'{metric_col}_screen']
monthly_merged['prof_pct'] = (monthly_merged[f'{metric_col}_prof'] / monthly_merged['total'] * 100).fillna(0)

current_month = pd.Timestamp.now().to_period('M').start_time
monthly_merged['is_current'] = monthly_merged['period_start'] == current_month

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Weekly plot
ax1 = axes[0]
ax2 = ax1.twinx()
colors = ['tab:orange' if is_curr else 'tab:blue' for is_curr in weekly_merged['is_current']]
ax1.bar(weekly_merged['period_start'], weekly_merged['prof_pct'],
        width=WEEKLY_BAR_WIDTH, alpha=0.7, color=colors, label='Prof %')
ax2.plot(weekly_merged.sort_values('period_start')['period_start'],
         weekly_merged.sort_values('period_start')['total']/1000,
         color='tab:red', marker='o', linewidth=2, markersize=4, label=f'Total {metric_display}')
ax1.set_ylabel('Professional %', fontsize=11, color='tab:blue')
ax2.set_ylabel(f'Total {metric_display} (000s)', fontsize=11, color='tab:red')
ax1.set_title(f'Weekly {metric_display}', fontsize=12, fontweight='bold')
ax1.set_xlabel('Date', fontsize=11)
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:red')
ax1.set_ylim(0, 110)
ax2.set_ylim(bottom=0)
ax1.grid(True, alpha=0.3, axis='y')
ax1.legend(loc='upper left', fontsize=9)
ax2.legend(loc='upper right', fontsize=9)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax1.set_xticks(_plot_months)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Monthly plot
ax1 = axes[1]
ax2 = ax1.twinx()
colors = ['tab:orange' if is_curr else 'tab:blue' for is_curr in monthly_merged['is_current']]
ax1.bar(monthly_merged['period_start'], monthly_merged['prof_pct'],
        width=MONTHLY_BAR_WIDTH, alpha=0.7, color=colors, label='Prof %')
ax2.plot(monthly_merged.sort_values('period_start')['period_start'],
         monthly_merged.sort_values('period_start')['total']/1000,
         color='tab:red', marker='o', linewidth=2, markersize=5, label=f'Total {metric_display}')
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax1.set_xticks(_plot_months)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

ax1.set_xlabel('Date', fontsize=11)
ax1.set_ylabel('Professional %', fontsize=11, color='tab:blue')
ax2.set_ylabel(f'Total {metric_display} (000s)', fontsize=11, color='tab:red')
ax1.set_title(f'Monthly {metric_display}', fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:red')
ax1.set_ylim(0, 110)
max_monthly_total = monthly_merged['total'].max() / 1000
ax2.set_ylim(0, max_monthly_total * 1.2)
ax1.grid(True, alpha=0.3, axis='y')
ax1.legend(loc='upper left', fontsize=9)
ax2.legend(loc='upper right', fontsize=9)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

plt.suptitle(f'Prof Trade % and Total {metric_display} - {FIGURE_TITLE_PORTFOLIO_NAME}',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
axes[0].set_xlim(_xlim_start_weekly, _xlim_end_weekly)
axes[1].set_xlim(_xlim_start_monthly, _xlim_end_monthly)
save_plot(fig, f'04_weekly_monthly_dual_axis_{FIGURE_TITLE_PORTFOLIO_NAME}')
plt.show()

# =============================================================================
# 10b. HELPER: BUILD COMMON COLUMNS USED BY PLOTS 11-12
# =============================================================================

monthly_merged['prof_pct_stack'] = (monthly_merged[f'{metric_col}_prof'] / monthly_merged['total'] * 100).fillna(0)
monthly_merged['screen_pct_stack'] = (monthly_merged[f'{metric_col}_screen'] / monthly_merged['total'] * 100).fillna(0)
weekly_merged['prof_pct_stack'] = (weekly_merged[f'{metric_col}_prof'] / weekly_merged['total'] * 100).fillna(0)
weekly_merged['screen_pct_stack'] = (weekly_merged[f'{metric_col}_screen'] / weekly_merged['total'] * 100).fillna(0)

portfolio_name = PORTFOLIO_NAME_FOR_OVERLAY.get(USER_SECTORS, PORTFOLIO_NAMES[0])
monthly_merged['year_month'] = monthly_merged['period_start'].dt.strftime('%Y-%m')

WIDE_MONTHLY_BAR_WIDTH = 22

# =============================================================================
# HELPER: reusable two-panel plot (weekly + monthly) with overlay line
# =============================================================================

def plot_absolute_bars_with_overlay(weekly_merged, monthly_merged, metric_col, metric_display,
                                    overlay_col, overlay_label, overlay_ylabel,
                                    suptitle, save_name, ylim_fallback=30):

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    outline = [pe.withStroke(linewidth=3, foreground='white')]

    ax = axes[0]
    ax.bar(weekly_merged['period_start'], weekly_merged[f'{metric_col}_prof'] / 1000,
           width=WEEKLY_BAR_WIDTH, alpha=0.8, color='tab:blue', label='Prof')
    ax.bar(weekly_merged['period_start'], weekly_merged[f'{metric_col}_screen'] / 1000,
           width=WEEKLY_BAR_WIDTH, alpha=0.8, color='tab:orange',
           bottom=weekly_merged[f'{metric_col}_prof'] / 1000, label='Screen')
    ax.set_ylabel(f'{metric_display} (000s)', fontsize=11)
    ax.set_xlabel('Date', fontsize=11)
    ax.set_title(f'Weekly {metric_display}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(loc='upper left', fontsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax.set_xticks(_plot_months)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    _plot_monthly_with_overlay(axes[1], monthly_merged, metric_col, metric_display,
                                overlay_col, overlay_label, overlay_ylabel,
                                MONTHLY_BAR_WIDTH, ylim_fallback, outline)

    plt.suptitle(suptitle, fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    axes[0].set_xlim(_xlim_start_weekly, _xlim_end_weekly)
    axes[1].set_xlim(_xlim_start_monthly, _xlim_end_monthly)
    save_plot(fig, save_name)
    plt.show()


# =============================================================================
# HELPER: reusable single-panel full-width monthly plot with overlay line
# =============================================================================

def plot_monthly_wide_with_overlay(monthly_merged, metric_col, metric_display,
                                    overlay_col, overlay_label, overlay_ylabel,
                                    suptitle, save_name, ylim_fallback=30):

    fig, ax1 = plt.subplots(1, 1, figsize=(14, 6))

    outline = [pe.withStroke(linewidth=3, foreground='white')]

    _plot_monthly_with_overlay(ax1, monthly_merged, metric_col, metric_display,
                                overlay_col, overlay_label, overlay_ylabel,
                                WIDE_MONTHLY_BAR_WIDTH, ylim_fallback, outline,
                                label_fontsize=9, overlay_fontsize=9)

    plt.suptitle(suptitle, fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    ax1.set_xlim(_xlim_start_monthly, _xlim_end_monthly)
    save_plot(fig, save_name)
    plt.show()


# =============================================================================
# HELPER: shared monthly panel logic (used by both two-panel and single-panel)
# =============================================================================

def _plot_monthly_with_overlay(ax1, monthly_merged, metric_col, metric_display,
                                overlay_col, overlay_label, overlay_ylabel,
                                bar_width, ylim_fallback, outline,
                                label_fontsize=7.5, overlay_fontsize=7.5):

    ax1.bar(monthly_merged['period_start'], monthly_merged[f'{metric_col}_prof'] / 1000,
           width=bar_width, alpha=0.8, color='tab:blue', label='Prof')
    ax1.bar(monthly_merged['period_start'], monthly_merged[f'{metric_col}_screen'] / 1000,
           width=bar_width, alpha=0.8, color='tab:orange',
           bottom=monthly_merged[f'{metric_col}_prof'] / 1000, label='Screen')

    for date, prof_val, total_val in zip(monthly_merged['period_start'],
                                          monthly_merged[f'{metric_col}_prof'],
                                          monthly_merged['total']):
        if total_val > 0:
            pct = prof_val / total_val * 100
            y_pos = prof_val / 1000
            ax1.text(date, y_pos + (total_val / 1000 * 0.015), f'{pct:.1f}%',
                    ha='center', va='bottom', fontsize=label_fontsize, fontweight='bold',
                    color='navy', path_effects=outline, zorder=6)

    ax2 = ax1.twinx()

    monthly_with_overlay = monthly_merged[monthly_merged[overlay_col].notna()]
    if not monthly_with_overlay.empty:
        ax2.plot(monthly_with_overlay['period_start'], monthly_with_overlay[overlay_col],
                color='red', linewidth=2.5, marker='o', markersize=6,
                label=overlay_label, zorder=5)

        vals = monthly_with_overlay[overlay_col].values
        n = len(vals)
        for i, (date, val) in enumerate(zip(monthly_with_overlay['period_start'], vals)):
            if n == 1:
                place_above = True
            elif i == 0:
                place_above = val <= vals[1]
            elif i == n - 1:
                place_above = val <= vals[i - 1]
            else:
                place_above = val <= vals[i - 1] or val <= vals[i + 1]

            if place_above:
                offset = 0.8
                va = 'bottom'
            else:
                offset = -0.8
                va = 'top'

            ax2.text(date, val + offset, f'{val:.1f}%',
                    ha='center', va=va, fontsize=overlay_fontsize, color='red',
                    fontweight='bold', path_effects=outline, zorder=7)

        ax2.set_ylabel(overlay_ylabel, fontsize=11, color='red')
        ax2.tick_params(axis='y', labelcolor='red')

        _ov_min = monthly_with_overlay[overlay_col].min()
        _ov_max = monthly_with_overlay[overlay_col].max()
        _pad = max((_ov_max - _ov_min) * 0.25, 3)
        ax2.set_ylim(
            min(_ov_min - _pad, -5) if _ov_min < 0 else max(_ov_min - _pad, 0),
            _ov_max + _pad if _ov_max + _pad > ylim_fallback else ylim_fallback
        )

    ax1.set_ylabel(f'{metric_display} (000s)', fontsize=11)
    ax1.set_xlabel('Date', fontsize=11)
    ax1.set_title(f'Monthly {metric_display} + {overlay_label}', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax1.set_xticks(_plot_months)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)


# =============================================================================
# 11. PLOT: ABSOLUTE STACKED BARS WITH MARGIN SHARE
# =============================================================================

ms_values = MS_DATA.get(portfolio_name, {})
monthly_merged['ms'] = monthly_merged['year_month'].map(ms_values)

plot_absolute_bars_with_overlay(
    weekly_merged, monthly_merged, metric_col, metric_display,
    overlay_col='ms',
    overlay_label='Margin Share',
    overlay_ylabel='MS (%)',
    suptitle=f'Prof/Screen {metric_display} + Margin Share - {FIGURE_TITLE_PORTFOLIO_NAME}',
    save_name=f'05_margin_share_participation_{FIGURE_TITLE_PORTFOLIO_NAME}',
    ylim_fallback=30,
)

plot_monthly_wide_with_overlay(
    monthly_merged, metric_col, metric_display,
    overlay_col='ms',
    overlay_label='Margin Share',
    overlay_ylabel='MS (%)',
    suptitle=f'Prof/Screen {metric_display} + Margin Share - {FIGURE_TITLE_PORTFOLIO_NAME}',
    save_name=f'05w_margin_share_wide_{FIGURE_TITLE_PORTFOLIO_NAME}',
    ylim_fallback=30,
)

# =============================================================================
# 11b. PLOT: ABSOLUTE STACKED BARS WITH MARGIN REVENUE
# =============================================================================

margin_rev_values = MARGIN_REV_DATA.get(portfolio_name, {})
monthly_merged['margin_rev'] = monthly_merged['year_month'].map(margin_rev_values)

plot_absolute_bars_with_overlay(
    weekly_merged, monthly_merged, metric_col, metric_display,
    overlay_col='margin_rev',
    overlay_label='Margin Rev',
    overlay_ylabel='Margin Rev (%)',
    suptitle=f'Prof/Screen {metric_display} + Margin Rev - {FIGURE_TITLE_PORTFOLIO_NAME}',
    save_name=f'05b_margin_rev_participation_{FIGURE_TITLE_PORTFOLIO_NAME}',
    ylim_fallback=30,
)

plot_monthly_wide_with_overlay(
    monthly_merged, metric_col, metric_display,
    overlay_col='margin_rev',
    overlay_label='Margin Rev',
    overlay_ylabel='Margin Rev (%)',
    suptitle=f'Prof/Screen {metric_display} + Margin Rev - {FIGURE_TITLE_PORTFOLIO_NAME}',
    save_name=f'05bw_margin_rev_wide_{FIGURE_TITLE_PORTFOLIO_NAME}',
    ylim_fallback=30,
)

# =============================================================================
# 12. PLOT: ABSOLUTE STACKED BARS WITH VEGA SHARE
# =============================================================================

vega_share_values = VEGA_SHARE_DATA.get(portfolio_name, {})
monthly_merged['vega_share'] = monthly_merged['year_month'].map(vega_share_values)

plot_absolute_bars_with_overlay(
    weekly_merged, monthly_merged, metric_col, metric_display,
    overlay_col='vega_share',
    overlay_label='Vega Share',
    overlay_ylabel='Vega Share (%)',
    suptitle=f'Prof/Screen {metric_display} + Vega Share - {FIGURE_TITLE_PORTFOLIO_NAME}',
    save_name=f'06_vega_share_participation_{FIGURE_TITLE_PORTFOLIO_NAME}',
    ylim_fallback=20,
)

plot_monthly_wide_with_overlay(
    monthly_merged, metric_col, metric_display,
    overlay_col='vega_share',
    overlay_label='Vega Share',
    overlay_ylabel='Vega Share (%)',
    suptitle=f'Prof/Screen {metric_display} + Vega Share - {FIGURE_TITLE_PORTFOLIO_NAME}',
    save_name=f'06w_vega_share_wide_{FIGURE_TITLE_PORTFOLIO_NAME}',
    ylim_fallback=20,
)

# =============================================================================
# 13. ANALYZE COUNTERPARTY NOTIONAL
# =============================================================================

print("\n" + "="*80)
print("COUNTERPARTY ANALYSIS")
print("="*80)

only_options = prof_trades_df[prof_trades_df['product_type']=='option']
df = only_options.dropna(subset=['counterpart'])

df['raw_notional'] = df['volume'] * df['price'] * df['multiplier']

def calculate_notional(group):
    if pd.isna(group['combo_group'].iloc[0]):
        return group['raw_notional'].sum()
    else:
        buy_notional = group[group['side'] == 'BUY']['raw_notional'].sum()
        sell_notional = group[group['side'] == 'SELL']['raw_notional'].sum()
        return abs(buy_notional - sell_notional)

df['group_id'] = df.groupby(['counterpart', 'combo_group']).ngroup()

notional_per_trade = df.groupby(['counterpart', 'group_id']).apply(calculate_notional).reset_index()
notional_per_trade.columns = ['counterpart', 'group_id', 'notional']

result = notional_per_trade.groupby('counterpart').agg(
    trade_count=('group_id', 'count'),
    total_notional=('notional', 'sum')
).reset_index()

result = result.sort_values('total_notional', ascending=False)

print("\nTop Counterparties by Notional:")
with pd.option_context('display.float_format', '{:,.2f}'.format):
    print(result.head(20).reset_index(drop=True))

print("\n" + "="*80)
print("DASHBOARD COMPLETE")
print("="*80)